# Jugalbandi — GRPO Training (Colab)

Fine-tunes Qwen2.5-0.5B-Instruct with GRPO against the Jugalbandi OpenEnv reward signal.

See `docs/Train.md` for the full walkthrough of every decision made here, and
`docs/EXPERIMENT_PLAN.md` for what this training run is actually being evaluated
against once it's done.

Training is **in-process and per-step** (updatedplan.md Phase 3, revised for Claim
B — in-context regime inference from reward feedback alone, §9 decision 2,
resolved in favour of Claim B): each GRPO sample is one decision point sampled
from a reference episode (which contains a drift event, or, for a 20% control
condition, no drift at all). The prompt shows the outcome of the *previous* step
(`raaga_env.prompting.StepFeedback`, Phase 2.3) — this is what gives an
otherwise-blind arm (DIAL/HIDDEN) any chance of noticing a rule change from
reward correlation alone, since without it, reward is never in the context at
all. The completion is one action; the reward is that action's immediate reward
plus a short Monte-Carlo continuation under a reference policy, computed by
restoring the exact env state the prompt was drawn from
(`JugalbandiEnv.set_state`, Phase 0.4) and replaying via
`eval.rollout.rollout_from_state` — the same function every baseline and
evaluation number goes through, so training reward and eval reward can never
silently diverge.

An earlier version of this notebook generated a full blind 64-action episode per
completion. That design can't use per-step feedback (a completion generated in
one shot can't react to feedback about steps it hasn't taken yet), so it was
replaced once Claim B was chosen — see `training/train_grpo.py`'s module
docstring for the full account.

This notebook is a thin driver over `training/train_grpo.py` — the pipeline logic
(dataset construction, drift scheduling, reward scoring) lives there and is
imported, not redefined here (Phase 3.4). Run cells top to bottom.

## Cell 1 — Install Dependencies

In [ ]:
# Takes ~3 minutes on Colab.
#
# This repo is public on GitHub, so the easiest path is just cloning it -
# no file upload needed. If that fails for any reason (offline mirror,
# renamed repo, you're testing local uncommitted changes), drop a raga.zip
# into the Files panel on the left (folder icon) before running this cell
# and it'll be used instead.
import os
import subprocess
import zipfile

REPO_URL = "https://github.com/jayyyyqwq/raga.git"
REPO_DIR = "raga"
ZIP_NAME = "raga.zip"

if not os.path.exists(REPO_DIR):
    if os.path.exists(ZIP_NAME):
        print(f"Found {ZIP_NAME} - extracting that instead of cloning.")
        with zipfile.ZipFile(ZIP_NAME) as zf:
            zf.extractall(".")
    else:
        print(f"Cloning {REPO_URL} ...")
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
        if result.returncode != 0:
            raise RuntimeError(
                "Clone failed. Drag a raga.zip file (see README's 'Advanced / "
                "manual version') into the Files panel on the left and re-run this cell."
            )
%cd {REPO_DIR}

# Installing straight from requirements-train.txt (not unsloth's own
# git+colab-new install line) - that line tries to build xformers from
# source against whatever CUDA/torch happens to be on the current Colab
# image, which breaks whenever Colab's image drifts from what it expects.
# requirements-train.txt pins unsloth itself and lets its own declared
# dependency ranges resolve everything else (see that file's header for
# why - it has broken Colab's "Run all" twice from over-pinning).
!pip install -r requirements-train.txt gymnasium numpy

# A failed `!pip install` doesn't raise inside a notebook cell, so
# without this check a broken install here surfaces several cells later
# as a confusing ModuleNotFoundError in the model-loading cell instead
# of at the point that actually failed. Fail loud, fail here.
import importlib.util

_required = ("unsloth", "torch", "trl", "peft", "transformers", "bitsandbytes")
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
if _missing:
    raise RuntimeError(
        f"pip install did not actually install: {_missing}. Scroll up in "
        "this cell's output for the real pip error - usually a version "
        "conflict, or a wheel not available for this Colab image's "
        "Python/CUDA. If unsloth itself is the problem, check whether a "
        "newer release exists: https://pypi.org/project/unsloth/#history "
        "and bump the pin in requirements-train.txt."
    )
print("All training deps installed OK.")


## Cell 2 — Mount Drive + Set Secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata


def _get_secret(name: str) -> str | None:
    """Both secrets below are optional — training runs fine with neither.
    Set one (key icon, left sidebar, toggle 'Notebook access' on) only if
    you want it: WANDB_API_KEY for live loss/reward charts, HF_TOKEN to push
    the final adapter to the HF Hub in Cell 7. Missing either one just skips
    that feature instead of crashing."""
    try:
        return userdata.get(name)
    except Exception:
        return None


wandb_key = _get_secret("WANDB_API_KEY")
hf_token = _get_secret("HF_TOKEN")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

print(f"wandb configured: {bool(wandb_key)}")
print(f"HF token configured: {bool(hf_token)}")

## Cell 3 — Load Model

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=512,  # one action per completion again — no need for episode-mode's headroom
    load_in_4bit=True,
    token=os.environ.get("HF_TOKEN"),  # fine as None — this is a public model
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(model.print_trainable_parameters())  # should say ~4M trainable params

## Cell 4 — Training Pipeline (imported from train_grpo.py)

Dataset construction, drift scheduling, and reward scoring all live in
`training/train_grpo.py` (Phase 3.4) — this notebook imports them rather than
redefining them, so it can't drift out of sync with the script the way the old
notebook's hand-duplicated prompt/reward code did (F7).

In [ ]:
from raaga_env.prompting import Arm
from training.train_grpo import (
    EPISODE_LENGTH,
    build_dataset,
    make_step_reward_fn,
    sample_drift_schedule,
)

# Which observation arm to train against. HIDDEN is the actual research
# question (updatedplan.md Phase 4.1) and the one worth training first if
# you're short on time — it's the arm nothing else in this project's history
# has ever measured. DIAL and ORACLE are comparison points, not required for
# a first working model. To train another arm, change this line and re-run
# from here down (Cells 4-8); RUN_NAME in Cell 6 picks up the arm name
# automatically so runs don't overwrite each other on Drive/wandb.
ARM = Arm.HIDDEN

In [ ]:
# GRPO scores one sampled action per completion: restore the env to the row's
# exact snapshot state (Phase 0.4), apply the action, then continue with a
# reference policy for a short horizon (Monte-Carlo return) — never a
# shared/mutated env instance across samples (Phase 3.1), never an unrelated
# freshly-reset episode (F1).
reward_fn = make_step_reward_fn(ARM)

## Cell 5 — Build Dataset

Each row is one decision point: walk a fresh reference episode forward under a
random-valid reference policy (with a sampled drift schedule — Phase 3.3:
drift_at_step uniform in [16, 48], flipped to the opposite raga, 20% no-switch
control condition), stop at a uniformly chosen step, and snapshot the prompt
(with `StepFeedback` from the immediately preceding step) plus the exact env
state. No HTTP (Phase 3.1).

Every CALL_EVERY steps (8, matching `JugalbandiEnv`), a raga-valid 4-swara call phrase is sampled and submitted via `env.set_call()` — this is new (ML retrain, 2026-10; see `docs/RETRAIN_PLAN.md`): training previously never called `set_call()` at all, so the model never saw a call phrase and the call-response reward layer scored against a phantom default call every step instead.

In [ ]:
# First time running this code (or after any dependency/version change —
# see train_grpo.py's usage comment): set STEPS = 50 for a pilot run first
# (a few minutes on a T4). Confirm it finishes, reward/loss log sensibly,
# and Cell 8's inference check runs — THEN come back and raise this for
# the real run. Don't go straight to a long run on code/deps you haven't
# seen complete once.
#
# 2026-10: the 50-step pilot already ran clean on this stack (see
# docs/RETRAIN_PLAN.md) — this is the real run now. 1000 steps on T4:
# roughly 2.5-3.5 hours (scaling from the v1 run's ~50 min / 300 steps).
# That's past Colab free tier's ~90-minute idle-disconnect window, so
# keep the tab open/active (an occasional click or scroll resets the
# idle timer) — interacting with the page is what matters, not watching
# it. If it does disconnect, Cell 6's save_total_limit=3 keeps the last
# 3 checkpoints on Drive to resume from instead of losing the whole run.
STEPS = 1000
BATCH = 4

print("Building training dataset...")
train_dataset = build_dataset(STEPS * BATCH, ARM)
print(f"Dataset size: {len(train_dataset)} decision points")
n_with_feedback = sum(1 for row in train_dataset if "Last action" in row["prompt"])
print(f"Rows with feedback (i.e. not step 0 of their episode): {n_with_feedback}/{len(train_dataset)}")
print(f"Sample prompt:\n{train_dataset[0]['prompt']}")

## Cell 6 — Training Config + Run

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# v2: the first run (v1, docs/FIRST_TRAINING_RUN.md) predates the call-
# response fix (docs/RETRAIN_PLAN.md) and stays on Drive as the
# documented baseline — bump this again for any run after this one.
RUN_NAME = f"jugalbandi-grpo-{ARM.value}-v2"

USE_WANDB = bool(os.environ.get("WANDB_API_KEY"))
if USE_WANDB:
    import wandb
    wandb.init(project="jugalbandi", name=RUN_NAME, config={"arm": ARM.value, "steps": STEPS, "batch": BATCH})
else:
    print("No WANDB_API_KEY secret set — training without wandb logging. "
          "Reward/loss still print to this cell's output every `logging_steps`.")

config = GRPOConfig(
    output_dir=f"/content/drive/MyDrive/jugalbandi/{RUN_NAME}",
    # No num_train_epochs: HF's Trainer lets a positive max_steps override
    # it, so setting it would be dead config implying something false.
    # Each optimizer step consumes BATCH * gradient_accumulation_steps
    # rows, so every row build_dataset() generated gets visited ~4 times
    # over a full run (K-epochs-per-rollout-batch, the same pattern PPO
    # uses) — see train_grpo.py's GRPOConfig comment for the full version.
    max_steps=STEPS,
    # per_device_train_batch_size * num_processes must be divisible by
    # num_generations (TRL GRPOTrainer constraint) — 2 % 4 != 0 would crash.
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    # Was save_steps=50 with no limit, sized for the earlier 200-300
    # step runs (6 checkpoints). At STEPS=1000 that would be 20 saves;
    # save_total_limit keeps only the latest 3 on Drive (each save is
    # the adapter + optimizer state, ~25-30MB going by the first run).
    save_steps=100,
    save_total_limit=3,
    report_to="wandb" if USE_WANDB else "none",
    num_generations=4,
    max_completion_length=8,  # one action index; 1-2 tokens
)

trainer = GRPOTrainer(
    model=model,
    args=config,
    reward_funcs=reward_fn,
    train_dataset=train_dataset,
    processing_class=tokenizer,        # `tokenizer=` is deprecated in TRL 0.12+
)

print(f"Starting training: {STEPS} steps, arm={ARM.value}")
trainer.train()
print("Training complete.")

**What to watch on wandb:**
- `train/reward_mean` should climb over 200+ steps
- `train/reward_std` collapsing to 0 means the model converged (possibly degenerate — check `safe_set_occupancy`-style behavior once Phase 4's eval harness is in place)
- Per-step rewards here are Monte-Carlo returns (immediate + a short reference-policy continuation), not single-step rewards — noisier than a pure bandit signal but far less noisy than scoring a full blind 64-action episode
- `grpo-HIDDEN` may simply fail to separate from a random-valid baseline at 0.5B — the risk register in updatedplan.md names this explicitly as a *possible, still publishable* result, not a bug, provided Phase 4's baselines (random-valid, safe-set-cycle, scripted-oracle) bracket it

## Cell 7 — Save and Push to Hub

In [ ]:
from huggingface_hub import HfApi

SAVE_DIR = f"/content/drive/MyDrive/jugalbandi/{RUN_NAME}/final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")
print("That's already in your Google Drive — open Drive in a browser afterward "
      "to see/download the adapter files. Nothing further is required.")

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    # CHANGE ME if jayyyyqwq isn't your Hugging Face username (GitHub handle
    # and HF username don't have to match) — check huggingface.co/settings/profile.
    api = HfApi()
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id="jayyyyqwq/jugalbandi-qwen-grpo",
        repo_type="model",
        token=hf_token,
        private=True,  # keep it off the public Hub index; drop this arg if you want it public later
    )
    print("Also pushed to HuggingFace Hub.")
else:
    print("No HF_TOKEN secret set — skipping the Hub push. "
          "Your model is already safe in Drive at SAVE_DIR above; that's all you need.")

## Cell 8 — Inference Check

Verify the model actually learned something before claiming results. This runs
a genuine multi-turn interactive episode: one model call per step, each prompt
showing the live `StepFeedback` from the step before — exactly the interaction
pattern the model was trained for (Phase 2.3 / Claim B), and the same pattern
`openenv_server/server.py`'s `/infer` + `/step` loop uses in production.

In [ ]:
from unsloth import FastLanguageModel
from raaga_env.jugalbandi_env import JugalbandiEnv
from raaga_env.prompting import StepFeedback, note_name, parse_action, render_prompt
from training.train_grpo import sample_call_phrase

model, tokenizer = FastLanguageModel.from_pretrained(SAVE_DIR, load_in_4bit=True)
FastLanguageModel.for_inference(model)

initial_dial, schedule = sample_drift_schedule()
schedule_by_step = schedule.as_dict()
env = JugalbandiEnv(initial_dial=initial_dial, episode_length=EPISODE_LENGTH)
obs, info = env.reset()

feedback = None
total_reward = 0.0
forbidden_count = 0
call_echo_count = 0

for step_idx in range(EPISODE_LENGTH):
    if step_idx in schedule_by_step:
        env.set_dial(schedule_by_step[step_idx])
        obs = env._get_obs()

    raga = env.drift.active_raga_name if ARM is Arm.ORACLE else None
    prompt = render_prompt(
        list(obs), arm=ARM, tala_pos=env.tala_position, feedback=feedback, raga=raga,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # do_sample=True is required here, not optional decoration: without it
    # HF's default is greedy decoding, which silently ignores temperature
    # and always picks the single most-likely token. Two checkpoints whose
    # underlying probabilities differ can still share the same argmax at
    # every step, producing a byte-identical episode (same drift schedule
    # too, from Cell 3's random_state=42) that looks like nothing changed
    # even when training genuinely did (hit for real, 2026-10, comparing a
    # 50-step pilot against a completed 1000-step run — see docs/RETRAIN_PLAN.md).
    # eval/llm_policy.py's make_llm_policy() already does this correctly
    # (do_sample=temperature > 0.0); this was the one call site that didn't.
    output = model.generate(**inputs, max_new_tokens=8, temperature=0.1, do_sample=True)
    decoded = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    action = parse_action(decoded)
    if action is None:
        print(f"Step {step_idx}: unparseable/out-of-range output {decoded!r} — stopping early.")
        break

    note, duration = env._decode(action)
    obs, reward, terminated, truncated, info = env.step(action)
    feedback = StepFeedback(note_name=note_name(note, duration), reward=float(reward))
    total_reward += reward
    if "forbidden_note" in info.get("reward_breakdown", {}):
        forbidden_count += 1
    if "call_echo" in info.get("reward_breakdown", {}):
        call_echo_count += 1
    if terminated or truncated:
        break

    # Same CALL_EVERY cadence build_dataset() trains against (ML retrain,
    # 2026-10; docs/RETRAIN_PLAN.md) — exercises the call-response path this
    # inference check previously never touched, since a solo improvisation
    # never submits a call.
    if info.get("call_requested"):
        env.set_call(sample_call_phrase(env.raga))
        obs = env._get_obs()

switch_step = schedule.switches[0][0] if schedule.switches else None
print(f"Drift schedule: initial={initial_dial}, switch_step={switch_step}")
print(f"Steps completed: {env.step_count}/{EPISODE_LENGTH}")
print(f"Episode return: {total_reward:.2f}")
print(f"Forbidden-note steps: {forbidden_count}/{env.step_count}")
print(f"Call-echo bonuses: {call_echo_count} (times the model landed on the partner's last call note)")